In [27]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

pd.set_option('display.max_columns', None)  # none은 제한값 자리(몇개 열까지 보여줄까)
pd.set_option('display.max_rows', None)

# ============================================================
# 0. 데이터 불러오기: 2-5 train_test_comparison에서 제거된 이상치까지 반영된 데이터셋
# ============================================================

train_full = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '0_train_full_clean.parquet')
test = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '0_test_future.parquet')
test_pred = pd.read_csv(PROJECT_ROOT.parent / 'data' / 'result_v1.csv')

from sklearn import set_config
set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드

In [2]:
print(test.shape, test_pred.shape)

(114111, 34) (186263, 2)


In [4]:
test_pred.head(5)
test_pred.info()

<class 'pandas.DataFrame'>
RangeIndex: 186263 entries, 0 to 186262
Data columns (total 2 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   row_id           186263 non-null  str    
 1   pred_maxrise_6h  186263 non-null  float64
dtypes: float64(1), str(1)
memory usage: 5.0 MB


In [29]:
# reindex(): 부모 인덱스 기준, 자식을 재정렬하여 가져옴
# 여러 컬럼을 한번에 지정해서 index에 맞게 다 들고 올 수 있음(map이랑 다른부분)
# reindex 용법: 부모[컬럼] = 자식[컬럼].reindex(부모인덱스)
# 부모, 자식 크기가 안맞을때 부모인덱스에 해당하는만큼만 자식에서 가져온다.
# 자식도 같은 인덱스가 지정 돼 있어야 한다.(set_index()를 써서라도)
test['test_pred_v1'] = (
    test_pred
    .set_index('row_id')['pred_maxrise_6h']  # 인덱스로 설정해야 매칭 가능
    # set_index로 인덱스 설정 -> pred_maxrise_6h 불러오기
    .reindex(test.index)
)

# 원본 예측값에 결측치가 없는지 확인
assert not test_pred['pred_maxrise_6h'].isnull().any(), '원본 예측값에 결측치 있음'

# reindex 후 미매칭 여부 확인
assert not test['test_pred_v1'].isnull().any(), '인덱스 미매칭 발생'


# map(): 시리즈나 딕셔너리, 함수로 매핑규칙을 지정, 부모와 자식의 인덱스가 같기만 하면
# 시리즈만 받으므로 여러 열을 가져오지 못한다.
# 매핑규칙으로 부모와 같은 인덱스와 값을 가진 자녀 객체가 있으면 됨

# map 용법: 부모[컬럼] = 부모 인덱스.map(매핑규칙(자녀의 것으로서))  # 부모 인덱스를 매핑규칙에 따라 자식 인덱스와 매칭
# 자식과 부모의 인덱스가 같은 것만 매핑 가능(set_index()를 써서라도)
# test['test_pred_v1'] = (
# test.index  # test는 'row_id'가 이미 인덱스라서 이렇게 들고옴
# .map(test_pred.set_index('row_id')['pred_maxrise_6h']) 
# )

In [7]:
print(test.shape, test_pred.shape)
test.head()

(114111, 35) (186263, 2)


,timestamp,station_id,wl_t,wl_t_minus_1h,wl_t_minus_3h,wl_t_minus_6h,wl_t_minus_12h,wl_t_minus_24h,imerg_rain_1h,imerg_rain_6h,...,wl_diff_3h,wl_diff_6h,wl_diff_12h,wl_diff_24h,imerg_int_1to6h,imerg_int_6to24h,sfc_int_1to6h,sfc_int_6to24h,target_maxrise_6h,test_pred_v1
row_id,,,,,,,,,,,,,,,,,,,,,
TE_000046543,2100-03-12,ST0001,0.19,0.20,0.22,0.25,0.16,0.18,2.333,2.340,...,-0.03,-0.06,0.03,0.01,0.0014,0.0,0.0,0.0,-0.01,0.099438
TE_000111013,2100-03-12,ST0002,0.85,0.84,0.86,0.85,0.84,0.85,0.140,0.140,...,-0.01,0.00,0.01,0.00,0.0000,0.0,0.0,0.0,0.01,0.008376
TE_000086280,2100-03-12,ST0003,0.76,0.76,0.77,0.78,0.77,0.76,2.772,2.773,...,-0.01,-0.02,-0.01,0.00,0.0002,0.0,0.0,0.0,0.01,0.042920
TE_000136298,2100-03-12,ST0005,0.98,0.96,0.96,0.93,0.95,0.97,0.205,0.208,...,0.02,0.05,0.03,0.01,0.0006,0.0,0.0,0.0,0.02,0.040083
TE_000166154,2100-03-12,ST0006,0.73,0.72,0.62,0.64,0.65,0.65,0.655,0.695,...,0.11,0.09,0.08,0.08,0.0080,0.0,0.0,0.0,0.02,0.047127


In [30]:
from sklearn.metrics import root_mean_squared_error

rmse = root_mean_squared_error(
    test['target_maxrise_6h'],
    test['test_pred_v1']
)

print(f'Test RMSE: {rmse:.6f}')

Test RMSE: 0.202310


In [10]:
# ============================================================
# 1. 컬럼군 정의
# ============================================================

target_col = 'target_maxrise_6h'  # 모델링 학습부터는 반드시 빼내야 함

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 수위차분(wl_diff) 컬럼군
wl_diff_cols = [
    'wl_t',
    'wl_diff_1h',
    'wl_diff_3h',
    'wl_diff_6h',
    'wl_diff_12h',
    'wl_diff_24h'
]


# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 강우강도 컬럼군
rain_int_cols = [
    'imerg_rain_1h',
    'imerg_int_1to6h',
    'imerg_int_6to24h',
    'sfc_rain_1h',
    'sfc_int_1to6h',
    'sfc_int_6to24h'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

num_cols = wl_cols + rain_cols + env_cols
num_cols_fe_wl = num_cols + [
    col for col in wl_diff_cols if col != 'wl_t'
]
num_cols_fe = num_cols_fe_wl + [
    col for col in rain_int_cols if col not in ['imerg_rain_1h', 'sfc_rain_1h']
]

print(f"""<변수개수 검증>
    수위변수: {len(wl_cols)}
    강우변수: {len(rain_cols)}, 
    환경변수: {len(env_cols)}, 
    수위차분(wl_t포함): {len(wl_diff_cols)},
    강우강도(rain_1h포함): {len(rain_int_cols)}
    기존 숫자열(수위 + 강우 + 환경): {len(num_cols)}, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): {len(num_cols_fe_wl)},
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): {len(num_cols_fe)}
""")

<변수개수 검증>
    수위변수: 6
    강우변수: 9, 
    환경변수: 7, 
    수위차분(wl_t포함): 6,
    강우강도(rain_1h포함): 6
    기존 숫자열(수위 + 강우 + 환경): 22, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): 27,
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): 31



In [17]:
# test 이상치 보정:  train, test 분포비교 수위변수 이상치
display(
    test.loc[
        (test['station_id'] == 'ST0436') &
        (
            (test['timestamp'].between('2102-12-26 00:00:00', '2102-12-28 12:00:00')) |
            (test['timestamp'].between('2103-11-02 00:00:00', '2103-11-03 00:00:00'))
        ),
    ['timestamp', 'station_id'] + wl_cols + [target_col] + ['test_pred_v1']
].sort_values('timestamp')
)

test_out_idx1 = test.loc[
    (test['station_id'] == 'ST0436') &
    (
        (test['timestamp'].between('2102-12-26 00:00:00', '2102-12-28 12:00:00')) |
        (test['timestamp'].between('2103-11-02 00:00:00', '2103-11-03 00:00:00'))
    )
].sort_values('timestamp').index

,timestamp,station_id,wl_t,wl_t_minus_1h,wl_t_minus_3h,wl_t_minus_6h,wl_t_minus_12h,wl_t_minus_24h,target_maxrise_6h,test_pred_v1
row_id,,,,,,,,,,
TE_000166886,2102-12-27 00:00:00,ST0436,-5.18,-5.18,-5.19,-5.19,-5.19,-5.19,0.03,0.050622
TE_000118763,2102-12-27 01:00:00,ST0436,-5.17,-5.18,-5.19,-5.19,-5.19,-5.18,0.03,0.049475
TE_000050700,2102-12-27 02:00:00,ST0436,-5.17,-5.17,-5.18,-5.19,-5.20,-5.17,0.04,0.035242
TE_000171285,2102-12-27 03:00:00,ST0436,-5.16,-5.17,-5.18,-5.19,-5.19,-5.17,0.03,0.025995
TE_000044147,2102-12-27 04:00:00,ST0436,-5.16,-5.16,-5.17,-5.19,-5.19,-5.17,0.04,0.009964
TE_000107495,2102-12-27 05:00:00,ST0436,-5.15,-5.16,-5.17,-5.18,-5.19,-5.17,0.04,0.030642
TE_000151241,2102-12-27 06:00:00,ST0436,-5.15,-5.15,-5.16,-5.18,-5.19,-5.17,0.04,0.020320
TE_000066551,2103-11-02 00:00:00,ST0436,0.07,0.06,0.07,0.03,0.06,0.28,-0.02,0.028642
TE_000006677,2103-11-02 01:00:00,ST0436,0.05,0.07,0.05,0.03,0.02,0.28,0.00,0.019465


In [13]:
# test 이상치 보정:  train, test 분포비교 수위변수 이상치
display(
    test.loc[
        (test['station_id'] == 'ST0158') &
        (test['timestamp'].between(
            '2102-12-26 07:00:00',
            '2102-12-28 07:00:00'
        )),
    ['timestamp', 'station_id'] + wl_cols + [target_col] + ['test_pred_v1']
].sort_values('timestamp')
)

,timestamp,station_id,wl_t,wl_t_minus_1h,wl_t_minus_3h,wl_t_minus_6h,wl_t_minus_12h,wl_t_minus_24h,target_maxrise_6h,test_pred_v1
row_id,,,,,,,,,,


In [18]:
# test 이상치 보정:  변수별 상관계수 분석에서 확인됨
cond_wl_out_station_vt1 = test['station_id'] == 'ST0445'
cond_wl_out_station_vt2 = test['station_id'] == 'ST0066'
cond_wl_out_ST0445_vt = (test[wl_cols] > 15).any(axis = 1)
cond_wl_out_ST0066_vt = (test[wl_cols] > 15).any(axis = 1)

display(
    test.loc[
        (
            (cond_wl_out_station_vt1 & cond_wl_out_ST0445_vt) |
            (cond_wl_out_station_vt2 & cond_wl_out_ST0066_vt)
        ),
        ['timestamp', 'station_id'] + wl_cols + [target_col] + ['test_pred_v1']
    ].sort_values('timestamp')
)

test_out_idx3 = test.loc[
    (
        (cond_wl_out_station_vt1 & cond_wl_out_ST0445_vt) |
        (cond_wl_out_station_vt2 & cond_wl_out_ST0066_vt)
    )
].sort_values('timestamp').index

,timestamp,station_id,wl_t,wl_t_minus_1h,wl_t_minus_3h,wl_t_minus_6h,wl_t_minus_12h,wl_t_minus_24h,target_maxrise_6h,test_pred_v1
row_id,,,,,,,,,,
TE_000138717,2102-02-20 00:00:00,ST0066,23.56,23.56,23.57,23.56,23.56,23.57,0.0,-0.006347
TE_000155915,2102-02-20 01:00:00,ST0066,23.56,23.56,23.56,23.56,23.56,23.57,0.0,-0.004560
TE_000085014,2102-02-20 02:00:00,ST0066,23.56,23.56,23.56,23.56,23.56,23.56,0.0,-0.006632


In [31]:
# 이상치 종속변수 0으로 변경 후 RMSE 산정

print(test_out_idx1)
print(test_out_idx3)

test_v2_out_correct = test.copy()
test_v2_out_correct.loc[test_out_idx1, 'test_pred_v1'] = 0
test_v2_out_correct.loc[test_out_idx3, 'test_pred_v1'] = 0

from sklearn.metrics import root_mean_squared_error

rmse = root_mean_squared_error(
    test_v2_out_correct['target_maxrise_6h'],
    test_v2_out_correct['test_pred_v1']
)

print(f'Test RMSE(이상치 1차 보정 후): {rmse:.6f}')

Index(['TE_000166886', 'TE_000118763', 'TE_000050700', 'TE_000171285',
       'TE_000044147', 'TE_000107495', 'TE_000151241', 'TE_000066551',
       'TE_000006677', 'TE_000046327', 'TE_000169718'],
      dtype='str', name='row_id')
Index(['TE_000138717', 'TE_000155915', 'TE_000085014'], dtype='str', name='row_id')
Test RMSE(이상치 1차 보정 후): 0.202310


In [37]:
# 최종 예측셋의 데이터별 잔차, SSE의 내림차순 정렬
# 잔차 (실제값 - 예측값)
test['residual'] = test['target_maxrise_6h'] - test['test_pred_v1']

# 잔차제곱 (각 행의 SSE 기여값)
test['residual_sq'] = test['residual'] ** 2

# 잔차 절대값 기준 내림차순 정렬
display(
    test.sort_values('residual', key=abs, ascending=False)[['station_id'] + rain_cols + [target_col] + ['test_pred_v1']].head(100)
)

# # 잔차제곱 기준 내림차순 정렬
# display(
#     test.sort_values('residual_sq', ascending=False)
# )

# 전체 SSE
print(f"SSE: {test['residual_sq'].sum():.6f}")

,station_id,imerg_rain_1h,imerg_rain_6h,imerg_rain_24h,sfc_rain_1h,sfc_rain_6h,sfc_rain_24h,radar_reflectivity_mean,radar_reflectivity_p95,radar_reflectivity_ge20_fraction,target_maxrise_6h,test_pred_v1
row_id,,,,,,,,,,,,
TE_000150989,ST0302,0.555,0.555,0.555,0.0,0.0,0.0,16.185,24.210,0.2471,3.42,0.033037
TE_000039994,ST0433,20.660,39.377,39.377,1.5,1.5,1.5,28.177,32.109,0.9824,3.16,0.078539
TE_000020895,ST0302,0.890,1.445,1.445,19.5,19.5,19.5,40.431,53.038,0.9258,3.43,0.370515
TE_000081035,ST0093,0.795,1.330,1.330,0.0,0.0,0.0,17.745,27.499,0.3789,3.11,0.057508
TE_000055832,ST0207,1.200,2.286,2.286,0.0,0.0,0.0,21.556,30.792,0.6992,3.14,0.095647
TE_000122659,ST0093,1.320,2.650,2.650,0.0,0.0,0.0,23.023,26.270,0.9189,3.10,0.061631
TE_000162662,ST0433,21.346,60.722,60.722,11.5,13.0,13.0,32.783,40.570,1.0000,3.27,0.234186
TE_000016542,ST0093,3.292,8.437,8.570,0.0,0.0,0.0,21.880,26.170,0.7637,3.10,0.088720
TE_000039179,ST0433,4.158,64.880,64.880,14.5,27.5,27.5,30.367,35.292,1.0000,3.27,0.261669


SSE: 4670.495803
